# 04 - CNN Story Point Regressor on BERT Token Embeddings (demo pair)

Three-layer 1D CNN over BERT token embeddings, trained to predict the story point.
This notebook runs one demo cross-project pair end to end (Nexus to Command-Line Interface);
05_run_all_pairs loops the same code over all 39 assigned pairs.

Pipeline per pair:
1. Train pool = source project's 80% (same seed-42 random split as the baseline).
2. 5-fold CV on the train pool: train on 4 folds, validate on 1 (early stopping). Reports mean +/- std val MAE.
3. Final model: train on the full train pool for the mean number of epochs the folds needed.
4. Evaluate once on the target project's held-out 20%.

Inputs: tawos-extracted dataset (splits + baseline comparison) and the output of
tawos-bert-embeddings (emb/*.npy, meta/*.csv). Needs GPU.

In [ ]:
import glob, os, re
import numpy as np, pandas as pd

csv = glob.glob("/kaggle/input/**/all_issues_sp.csv", recursive=True)[0]
df = pd.read_csv(csv)
df["storypoint"] = pd.to_numeric(df["storypoint"], errors="coerce")
df = df.dropna(subset=["storypoint"])
df = df[(df.storypoint > 0) & (df.storypoint <= 40)]
df["text"] = df["text"].fillna("").astype(str)
df["pos"] = np.arange(len(df))          # position in this filtered frame == row order of meta/*.csv

# identical split construction to 02_baseline_median (seed 42, same rng consumption order).
# baseline splits hold global filtered-frame indices; embeddings are per-project rows,
# so we also store each selection as local row ranks within its project.
rng = np.random.default_rng(42)
splits = {}
for proj, g in df.groupby("project"):
    idx = g.index.to_numpy()
    rng.shuffle(idx)
    n_test = max(1, int(round(0.2 * len(idx))))
    local = np.arange(len(g))
    lmap = dict(zip(g.index.to_numpy().tolist(), local.tolist()))
    splits[proj] = dict(test=np.array([lmap[v] for v in idx[:n_test]]),
                        train=np.array([lmap[v] for v in idx[n_test:]]))

EMB = glob.glob("/kaggle/input/**/emb", recursive=True)[0]
META = EMB.replace("/emb", "/meta")
def safe(n):
    return re.sub(r"[^A-Za-z0-9_]", "_", str(n))

def load_project(name):
    e = np.load(f"{EMB}/{safe(name)}.npy", mmap_mode="r")
    m = pd.read_csv(f"{META}/{safe(name)}.csv")
    assert len(e) == len(m)
    return e, m

e, m = load_project("Sonatype Nexus")
print("Nexus emb:", e.shape, "| meta rows:", len(m), "| test n:", len(splits["Sonatype Nexus"]["test"]))
# sanity: test-set sizes must match what the baseline recorded for each project
base = pd.read_csv(glob.glob("/kaggle/input/**/baseline_results.csv", recursive=True)[0])
within = base[base.target.str.contains("WITHIN")].set_index("source")
for proj in list(splits)[:5]:
    assert len(splits[proj]["test"]) == int(within.loc[proj, "n_test"]), proj
print("split sizes match baseline for sampled projects")


In [ ]:
import torch, torch.nn as nn

MAX_LEN, D = 128, 768

class CNNRegressor(nn.Module):
    # 3-layer 1D CNN over token embeddings + masked max pool + MLP head
    def __init__(self, d_in=D, ch=(256, 128, 64), p=0.2):
        super().__init__()
        c1, c2, c3 = ch
        self.conv = nn.Sequential(
            nn.Conv1d(d_in, c1, 3, padding=1), nn.ReLU(), nn.Dropout(p),
            nn.Conv1d(c1, c2, 3, padding=1), nn.ReLU(),
            nn.Conv1d(c2, c3, 3, padding=1), nn.ReLU(),
        )
        self.head = nn.Sequential(nn.Linear(c3, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x, ntok):                 # x [B, L, D] float32, ntok [B]
        h = self.conv(x.transpose(1, 2))        # [B, C, L]
        mask = (torch.arange(x.size(1), device=x.device)[None, :] < ntok[:, None]).unsqueeze(1)
        h = h.masked_fill(~mask, float("-inf"))
        return self.head(h.max(dim=2).values).squeeze(-1)

def batch_to_gpu(X, y, ntok, dev="cuda"):
    return (torch.tensor(X, dtype=torch.float32, device=dev),
            torch.tensor(y, dtype=torch.float32, device=dev),
            torch.tensor(ntok, dtype=torch.long, device=dev))

def train_model(Xtr, ytr, ntr, epochs=25, bs=64, lr=1e-3, patience=4, seed=0, Xva=None, yva=None, nva=None):
    torch.manual_seed(seed); np.random.seed(seed)
    model = CNNRegressor().cuda()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.L1Loss()
    n = len(ytr); best = (1e9, None, 0); hist = []
    for ep in range(epochs):
        model.train()
        perm = np.random.permutation(n)
        for b0 in range(0, n, bs):
            sel = perm[b0:b0 + bs]
            x, y, nt = batch_to_gpu(Xtr[sel], ytr[sel], ntr[sel])
            with torch.autocast("cuda", dtype=torch.float16):
                loss = lossf(model(x, nt), y)
            opt.zero_grad(); loss.backward(); opt.step()
        if Xva is not None:
            model.eval(); errs = []
            with torch.no_grad():
                for b0 in range(0, len(yva), 256):
                    x, y, nt = batch_to_gpu(Xva[b0:b0+256], yva[b0:b0+256], nva[b0:b0+256])
                    errs.append((model(x, nt) - y).abs().cpu().numpy())
            vmae = float(np.concatenate(errs).mean()); hist.append(vmae)
            if vmae < best[0]: best = (vmae, {k: t.clone() for k, t in model.state_dict().items()}, ep)
            if ep - best[2] >= patience: break
    if Xva is not None: model.load_state_dict(best[1])
    return model, hist

def predict(model, X, ntok, bs=512):
    model.eval(); out = []
    with torch.no_grad():
        for b0 in range(0, len(X), bs):
            x = torch.tensor(X[b0:b0+bs], dtype=torch.float32, device="cuda")
            nt = torch.tensor(ntok[b0:b0+bs], dtype=torch.long, device="cuda")
            out.append(model(x, nt).cpu().numpy())
    return np.concatenate(out)

print("model ready")

In [ ]:
from sklearn.model_selection import KFold

SRC, TGT = "Sonatype Nexus", "Command-Line Interface"
SEED = 42

Xs, ms = load_project(SRC); Xt, mt = load_project(TGT)
tr_pos, te_pos_src = splits[SRC]["train"], splits[TGT]["test"]
Xtr_all, ytr_all = Xs[tr_pos], ms.storypoint.values[tr_pos]
ntr_all = np.minimum(ms.n_tokens.values[tr_pos], MAX_LEN)

# 5-fold CV on the source train pool
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
fold_maes, best_eps = [], []
for f, (a, b) in enumerate(kf.split(np.arange(len(ytr_all)))):
    mdl, hist = train_model(Xtr_all[a], ytr_all[a], ntr_all[a], seed=SEED + f,
                            Xva=Xtr_all[b], yva=ytr_all[b], nva=ntr_all[b])
    fold_maes.append(hist[-1]); best_eps.append(len(hist))
    print(f"fold {f+1}: val MAE {hist[-1]:.3f} (stopped ep {len(hist)})", flush=True)
val_mean, val_std = float(np.mean(fold_maes)), float(np.std(fold_maes))
fix_epochs = int(np.clip(round(np.mean(best_eps)), 5, 25))

# final model on the full train pool, fixed epochs (no peeking at the target)
mdl, _ = train_model(Xtr_all, ytr_all, ntr_all, epochs=fix_epochs, seed=SEED)
pred = predict(mdl, Xt[te_pos_src], np.minimum(mt.n_tokens.values[te_pos_src], MAX_LEN))
ytest = mt.storypoint.values[te_pos_src]
err = np.abs(pred - ytest)
test_mae, test_mdae = float(err.mean()), float(np.median(err))

base = pd.read_csv(glob.glob("/kaggle/input/**/baseline_results.csv", recursive=True)[0])
brow = base[(base.source == SRC) & (base.target == TGT)]
bmae = float(brow.mae.iloc[0]) if len(brow) else float("nan")

print("")
print(f"{SRC} -> {TGT}")
print(f"5-fold val MAE: {val_mean:.3f} +/- {val_std:.3f} | fixed epochs for final: {fix_epochs}")
print(f"test MAE {test_mae:.3f} (baseline {bmae:.3f}) | test MdAE {test_mdae:.3f}")
pd.DataFrame([dict(source=SRC, target=TGT, val_mae_mean=val_mean, val_mae_std=val_std,
                   epochs=fix_epochs, test_mae=test_mae, test_mdae=test_mdae, baseline_mae=bmae)]
             ).to_csv("/kaggle/working/demo_pair_result.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].scatter(ytest, pred, s=14, alpha=0.6)
lims = [0, max(ytest.max(), pred.max()) + 1]
ax[0].plot(lims, lims, "r--"); ax[0].set_xlabel("actual"); ax[0].set_ylabel("predicted")
ax[0].set_title(f"{SRC} -> {TGT}")
ax[1].hist(err, bins=25); ax[1].set_title("abs error"); ax[1].set_xlabel("|pred - actual|")
plt.tight_layout(); plt.savefig("/kaggle/working/demo_pair_plots.png", dpi=120); plt.show()